In [ ]:
import sys
import os

import numpy as np
from ripser import ripser
from persim import plot_diagrams


from opt_einsum import contract
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
# from diffusion_geometry.src.visualisation import *
from plotly.subplots import make_subplots
# from diffusion_geometry import Function, VectorField, Form
from generate_data import gen_2d_data
# from diffusion_geometry import DiffusionGeometry
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
from matplotlib.path import Path
import pandas as pd
import matplotlib.pyplot as plt
from figures.generate_data import gen_3d_data
import time
import numpy as np
from diffusion_geometry.core.geometry.diffusion_geometry import DiffusionGeometry
from diffusion_geometry.visualisation import plot_scatter_3d


import matplotlib as mpl

mpl.rcParams.update(mpl.rcParamsDefault) 
plt.rcParams['font.family'] = 'serif'
mpl.rcParams['mathtext.fontset'] = 'cm'



dot_colors = ["#e32636","#166dde", "#d3d3d3",]

In [ ]:


def sample_torus(minor_radius, major_radius, n, rng):
    theta = rng.uniform(0.0, 2.0 * np.pi, n)
    phi   = rng.uniform(0.0, 2.0 * np.pi, n)

    r = minor_radius
    R = major_radius

    x = (R + r * np.cos(theta)) * np.cos(phi)
    y = (R + r * np.cos(theta)) * np.sin(phi)
    z = r * np.sin(theta)

    return np.column_stack((x, y, z))


def add_uniform_noise(torus_data, rng, n_extra_points):
    # torus_data = sample_torus(minor_radius, major_radius, n, seed)
    min_x, max_x = np.min(torus_data[:,0]), np.max(torus_data[:,0])
    min_y, max_y = np.min(torus_data[:,1]), np.max(torus_data[:,1])
    min_z, max_z = np.min(torus_data[:,2]), np.max(torus_data[:,2])

    # Add uniformely sampled points on the bounding box to improve visualization of vector fields
    extra_x = np.random.uniform(min_x, max_x, n_extra_points)
    extra_y = np.random.uniform(min_y, max_y, n_extra_points)
    extra_z = np.random.uniform(min_z, max_z, n_extra_points)
    extra_points = np.vstack((extra_x, extra_y, extra_z)).T

    all_data = np.vstack((torus_data, extra_points))
    return all_data, extra_points


def add_isotropic_noise(base_data, noise_level, rng):
    # rng = np.random.default_rng(seed)
    # base_data = sample_torus(minor_radius, major_radius, n, rng)
    noise = rng.normal(scale=noise_level, size=base_data.shape)
    perturbed_data = base_data + noise
   
    return perturbed_data

# ============================================================
# Diffusion geometry (computed ONCE per dataset)
# ============================================================

def compute_hodge_modes(data):
    dg = DiffusionGeometry.from_point_cloud(
        data,
        knn_kernel=32,
        n0=100,
    )

    UH1 = dg.up_laplacian(1)
    DH1 = dg.down_laplacian(1)
    H1 = UH1 + 1e10 * DH1

    eigvals, forms = H1.spectrum()
    print("Eigenvalues:", eigvals[:10])

    return forms[0], forms[1]


In [ ]:



from diffusion_geometry.visualisation import clean_fig, plot_quiver_3d


num_rows = 3
num_cols = 3

specs = [
    [{"type": "scene"}]*num_cols,
    [{"type": "scene"}]*num_cols,
    [{"type": "scene"}]*num_cols,
]

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.05,
    shared_xaxes=False,
    shared_yaxes=False
)

class Params:
    def __init__(self, quiver_scale, arrow_scale, line_width):
        self.quiver_scale=quiver_scale
        self.arrow_scale=arrow_scale
        self.line_width=line_width

def plot_column(fig, data, column,params_1, params_2):
    fig1 = plot_scatter_3d(data, size=2)
    fig.add_traces(list(fig1.data), rows=1, cols=column)

    dg = DiffusionGeometry.from_point_cloud(data, knn_kernel=32,n0=100)

    UH1 = dg.up_laplacian(1)
    DH1 = dg.down_laplacian(1)
    H1 = UH1 + 1e10 * DH1
    eig_values, forms_1 = H1.spectrum()


    # eig_values, forms_1 = dg.laplacian(1).spectrum(return_eigenvectors=True)
    print(f"Eigenvalues for 1-forms for column {column}:", eig_values[:10])

    quiver_scale = params_1.quiver_scale
    arrow_scale = params_1.arrow_scale
    line_width = params_1.line_width

    a = forms_1[0]
    b = forms_1[1]

    fig2 = plot_quiver_3d(data, a.to_ambient(), scale=quiver_scale, arrow_scale=arrow_scale, line_width=line_width)
    fig.add_traces(list(fig2.data), rows=2, cols=column)

    quiver_scale = params_2.quiver_scale
    arrow_scale = params_2.arrow_scale
    line_width = params_2.line_width

    fig3 = plot_quiver_3d(data, b.to_ambient(), scale=quiver_scale, arrow_scale=arrow_scale, line_width=line_width)
    fig.add_traces(list(fig3.data), rows=3, cols=column)

#seed = 4 for 1.5k is good

seed = 4
minor_radius = 1.0
major_radius = 2.5

rng = np.random.default_rng(seed)

params1 = Params(quiver_scale = 0.3, arrow_scale = 0.9,line_width = 0.7)
params2 = Params(quiver_scale = 0.3, arrow_scale = 1.1,line_width = 0.7)
params3 = Params(quiver_scale = 0.3, arrow_scale = 0.6,line_width = 0.7)
params4 = Params(quiver_scale = 0.3, arrow_scale = 1.1,line_width = 0.7)
params5 = Params(quiver_scale = 0.3, arrow_scale = 0.8,line_width = 0.7)
params6 = Params(quiver_scale = 0.3, arrow_scale = 1.1,line_width = 0.7)


clean_torus_data = sample_torus(minor_radius, major_radius, 1500, rng)
plot_column(fig, clean_torus_data, 1,params1,params2)

noise_level = 0.2
perturbed_torus_data = add_isotropic_noise(clean_torus_data, noise_level, rng)
plot_column(fig, perturbed_torus_data, 2,params3,params4)

n_extra_points = 100
noisy_torus, added_points = add_uniform_noise(clean_torus_data, rng, n_extra_points)
extra_fig = plot_scatter_3d(added_points, size=2, color=dot_colors[0])


plot_column(fig, noisy_torus, 3,params5,params6)
fig.add_traces(list(extra_fig.data), rows=1, cols=3)




camera = dict(
    eye=dict(x=0, y=1.2, z=1.3),  
    up=dict(x=0, y=0, z=1),     
    center=dict(x=0, y=0.1, z=0)
)

all_data = np.vstack([clean_torus_data, perturbed_torus_data])
x_min, x_max = all_data[:,0].min(), all_data[:,0].max()
y_min, y_max = all_data[:,1].min(), all_data[:,1].max()
z_min, z_max = all_data[:,2].min(), all_data[:,2].max()

zoom_factor = 0.9  # 0.5 = zoom in 2x, 2.0 = zoom out 2x

x_center = (x_max + x_min)/2
y_center = (y_max + y_min)/2
z_center = (z_max + z_min)/2

x_range = (x_max - x_min) * zoom_factor / 2
y_range = (y_max - y_min) * zoom_factor / 2
z_range = (z_max - z_min) * zoom_factor / 2

for i in range(1, num_rows * num_cols + 1):
    scene_key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        scene_key: dict(
            camera=camera,
            aspectmode="data",
            # xaxis=dict(range=[x_center - x_range, x_center + x_range]),
            # yaxis=dict(range=[y_center - y_range, y_center + y_range]),
            # zaxis=dict(range=[z_center - z_range, z_center + z_range]),
        )
    })

fig.update_layout(width=1000, height=1000) 


clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=10))
fig.show()




In [ ]:
import matplotlib.pyplot as plt

# This function is modified from ripser.py
# Original authors:
#   - Ulrich Bauer (Ripser core)
#   - Christopher Tralie and Nathaniel Saul (ripser.py)
# Licensed under the MIT License.
# Modifications by David Lanners.

def my_plot_diagrams(
    diagrams,
    plot_only=None,
    title=None,
    xy_range=None,
    labels=None,
    colormap="default",
    size=20,
    ax_color=np.array([0.0, 0.0, 0.0]),
    diagonal=True,
    lifetime=False,
    legend=True,
    show=False,
    ax=None,
    colors=None,
    file_path=None,
    label_size=20,
    ticks_size=12,
    legend_size=20,
    ellipse=None,
):
    """A helper function to plot persistence diagrams. 

    Parameters
    ----------
    diagrams: ndarray (n_pairs, 2) or list of diagrams
        A diagram or list of diagrams. If diagram is a list of diagrams, 
        then plot all on the same plot using different colors.
    plot_only: list of numeric
        If specified, an array of only the diagrams that should be plotted.
    title: string, default is None
        If title is defined, add it as title of the plot.
    xy_range: list of numeric [xmin, xmax, ymin, ymax]
        User provided range of axes. This is useful for comparing 
        multiple persistence diagrams.
    labels: string or list of strings
        Legend labels for each diagram. 
        If none are specified, we use H_0, H_1, H_2,... by default.
    colormap: string, default is 'default'
        Any of matplotlib color palettes. 
        Some options are 'default', 'seaborn', 'sequential'. 
        See all available styles with

        .. code:: python

            import matplotlib as mpl
            print(mpl.styles.available)

    size: numeric, default is 20
        Pixel size of each point plotted.
    ax_color: any valid matplotlib color type. 
        See [https://matplotlib.org/api/colors_api.html](https://matplotlib.org/api/colors_api.html) for complete API.
    diagonal: bool, default is True
        Plot the diagonal x=y line.
    lifetime: bool, default is False. If True, diagonal is turned to False.
        Plot life time of each point instead of birth and death. 
        Essentially, visualize (x, y-x).
    legend: bool, default is True
        If true, show the legend.
    show: bool, default is False
        Call plt.show() after plotting. If you are using self.plot() as part 
        of a subplot, set show=False and call plt.show() only once at the end.
    """

    ax = ax or plt.gca()
    plt.style.use(colormap)

    xlabel, ylabel = "birth", "death"

    if not isinstance(diagrams, list):
        # Must have diagrams as a list for processing downstream
        diagrams = [diagrams]

    if labels is None:
        # Provide default labels for diagrams if using self.dgm_
        labels = [r"$H_{{{}}}$".format(i) for i , _ in enumerate(diagrams)]

    if plot_only:
        diagrams = [diagrams[i] for i in plot_only]
        labels = [labels[i] for i in plot_only]

    if not isinstance(labels, list):
        labels = [labels] * len(diagrams)

    # Construct copy with proper type of each diagram
    # so we can freely edit them.
    diagrams = [dgm.astype(np.float32, copy=True) for dgm in diagrams]

    # find min and max of all visible diagrams
    concat_dgms = np.concatenate(diagrams).flatten()
    has_inf = np.any(np.isinf(concat_dgms))
    finite_dgms = concat_dgms[np.isfinite(concat_dgms)]

    # clever bounding boxes of the diagram
    if not xy_range:
        # define bounds of diagram
        ax_min, ax_max = np.min(finite_dgms), np.max(finite_dgms)
        x_r = ax_max - ax_min

        # Give plot a nice buffer on all sides.
        # ax_range=0 when only one point,
        buffer = 1 if xy_range == 0 else x_r / 5

        x_down = ax_min - buffer / 2
        x_up = ax_max + buffer

        y_down, y_up = x_down, x_up
    else:
        x_down, x_up, y_down, y_up = xy_range

    yr = y_up - y_down

    if lifetime:

        # Don't plot landscape and diagonal at the same time.
        diagonal = False

        # reset y axis so it doesn't go much below zero
        y_down = -yr * 0.05
        y_up = y_down + yr

        # set custom ylabel
        ylabel = "Lifetime"

        # set diagrams to be (x, y-x)
        for dgm in diagrams:
            dgm[:, 1] -= dgm[:, 0]

        # plot horizon line
        ax.plot([x_down, x_up], [0, 0], c=ax_color)

    # Plot diagonal
    # if diagonal:
    #     ax.plot([x_down, x_up], [x_down, x_up], "--", c=ax_color)

    # Plot inf line
    # if has_inf:
    #     # put inf line slightly below top
    #     b_inf = y_down + yr * 0.95
    #     #plot as dotted line
    #     ax.plot([x_down, x_up], [b_inf, b_inf], ":", c="k")#, label=r"$\infty$")

    #     # convert each inf in each diagram with b_inf
    #     for dgm in diagrams:
    #         dgm[np.isinf(dgm)] = b_inf

    # Plot each diagram
    for i, (dgm, label) in enumerate(zip(diagrams, labels)):

        # plot persistence pairs
        ax.scatter(dgm[:, 0], dgm[:, 1], size, label=label, edgecolor="none", color = colors[i])

        ax.set_xlabel(xlabel, fontsize=label_size)
        ax.set_ylabel(ylabel, fontsize=label_size)

    ax.set_xlim([x_down, x_up])
    ax.set_ylim([y_down, y_up])
    ax.set_aspect('equal', 'box')

    ax.tick_params(axis='both', which='major', labelsize=ticks_size)
    
    from matplotlib.patches import Ellipse
    if ellipse is not None:
        ax.add_patch(ellipse)


    if title is not None:
        ax.set_title(title)

    if legend is True:
        ax.legend(loc="lower right", fontsize=legend_size)

    # if file_path is not None:
    #     plt.tight_layout()
    #     plt.savefig(file_path)

    if show is True:
        plt.show()

    return ax



In [ ]:
seed = 4
rng = np.random.default_rng(seed)

minor_radius = 1.0
major_radius = 2.5

clean_torus_data = sample_torus(
    minor_radius,
    major_radius,
    1500,
    rng,
)

noise_level = 0.2
perturbed_torus_data = add_isotropic_noise(
    clean_torus_data,
    noise_level,
    rng,
)

n_extra_points = 100
noisy_torus_data, added_points = add_uniform_noise(
    clean_torus_data,
    rng,
    n_extra_points,
)

diagram_clean = ripser(clean_torus_data, maxdim=1)['dgms']
diagram_numerical = ripser(perturbed_torus_data, maxdim=1)['dgms']
diagram_statistical = ripser(noisy_torus_data, maxdim=1)['dgms']



In [ ]:
from matplotlib.patches import Ellipse

label_size = 24
ticks_size = 22
legend_size = 23
size=50

ellipse = Ellipse(
                xy=(0.3, 2.15), 
                width=0.65, 
                height=1.4, 
                angle=0, 
                edgecolor=dot_colors[0], 
                fc='none',      # Transparent fill
                lw=2,           # Line width
                linestyle='--'
            )

ellipse2 = Ellipse(
                xy=(0.35, 1.55), 
                width=0.4, 
                height=1.1, 
                angle=0, 
                edgecolor=dot_colors[0], 
                fc='none',      # Transparent fill
                lw=2,           # Line width
                linestyle='--'
            )

colors = [dot_colors[1], dot_colors[0]]

# file_path="../figures/figs/resistance_to_noise/clean_pd.pdf"
fig1, ax1 = plt.subplots()
d1 = my_plot_diagrams(diagram_clean, ax=ax1, ellipse=ellipse, size=size, colors=colors, label_size=label_size, ticks_size=ticks_size, legend_size=legend_size)
# fig1.show()

fig2, ax2 = plt.subplots()
d2 = my_plot_diagrams(diagram_numerical, ax=ax2, ellipse=ellipse2, size=size, colors=colors, label_size=label_size, ticks_size=ticks_size, legend_size=legend_size)
# fig2.show()

fig3, ax3 = plt.subplots()
d3 = my_plot_diagrams(diagram_statistical, ax=ax3, size=size, colors=colors, label_size=label_size, ticks_size=ticks_size, legend_size=legend_size)
# fig3.show()

xlim1 = ax1.get_xlim()
ylim1 = ax1.get_ylim()
xlim2 = ax2.get_xlim()
ylim2 = ax2.get_ylim()
xlim3 = ax3.get_xlim()
ylim3 = ax3.get_ylim()

common_xmin = min(xlim1[0], xlim2[0], xlim3[0])
common_xmax = max(xlim1[1], xlim2[1], xlim3[1])
common_ymin = min(ylim1[0], ylim2[0], ylim3[0])
common_ymax = max(ylim1[1], ylim2[1], ylim3[1])

ax1.set_xlim(common_xmin, common_xmax)
ax1.set_ylim(common_ymin, common_ymax)
ax2.set_xlim(common_xmin, common_xmax)
ax2.set_ylim(common_ymin, common_ymax)
ax3.set_xlim(common_xmin, common_xmax)
ax3.set_ylim(common_ymin, common_ymax)

# draw diagonals
for ax in [ax1, ax2, ax3]:
    ax.plot([common_xmin, common_xmax], [common_xmin, common_xmax], "--", c='black')


# save figures
# fig1.savefig("../figures/figs/resistance_to_noise/clean_pd.pdf", bbox_inches='tight')
# fig2.savefig("../figures/figs/resistance_to_noise/numerical_pd.pdf", bbox_inches='tight')
# fig3.savefig("../figures/figs/resistance_to_noise/statistical_pd.pdf", bbox_inches='tight')

# my_plot_diagrams(diagram_numerical, show=True, ellipse=ellipse2, size=size, colors=dot_colors, file_path="../figures/figs/resistance_to_noise/numerical_pd.pdf", label_size=label_size, ticks_size=ticks_size, legend_size=legend_size)
# my_plot_diagrams(diagram_statistical, show=True, size=size, colors=dot_colors, file_path="../figures/figs/resistance_to_noise/statistical_pd.pdf", label_size=label_size, ticks_size=ticks_size, legend_size=legend_size)